In [1]:
import pandas as pd
import requests
from io import StringIO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import kaleido
import numpy as np
from dotenv import load_dotenv
import os

In [2]:
response = requests.get('https://drive.usercontent.google.com/u/0/uc?id=1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ&export=download')
reg_data_csv = pd.read_csv(StringIO(response.text))
response = requests.get('https://drive.usercontent.google.com/u/0/uc?id=1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe&export=download')
visits_data_csv = pd.read_csv(StringIO(response.text))

In [3]:
reg_data_csv.describe

<bound method NDFrame.describe of                     date  user_id                    email platform  \
0    2023-03-01T00:25:39  8838849     joseph95@example.org      web   
1    2023-03-01T14:53:01  8741065  janetsuarez@example.net      web   
2    2023-03-01T14:27:36  1866654     robert67@example.org      web   
3    2023-03-01T02:42:34  1577584         elam@example.net      web   
4    2023-03-01T10:27:14  4765395  stephanie68@example.net      web   
..                   ...      ...                      ...      ...   
995  2023-03-05T03:19:40  6414793  jerryrivera@example.com  android   
996  2023-03-05T15:03:48    53167      laura65@example.net  android   
997  2023-03-05T09:51:08  4969623    jasmine42@example.net  android   
998  2023-03-05T16:25:59  7636873  imclaughlin@example.net  android   
999  2023-03-05T11:17:41  7551697  jacobgarcia@example.org  android   

    registration_type  
0              google  
1              yandex  
2              google  
3               a

In [4]:
visits_data_csv.describe

<bound method NDFrame.describe of                                      uuid platform  \
0    1de9ea66-70d3-4a1f-8735-df5ef7697fb9      web   
1    f149f542-e935-4870-9734-6b4501eaf614      web   
2    f149f542-e935-4870-9734-6b4501eaf614      web   
3    08f0ebd4-950c-4dd9-8e97-b5bdf073eed1      web   
4    08f0ebd4-950c-4dd9-8e97-b5bdf073eed1      web   
..                                    ...      ...   
995  8bc047ac-dc03-4a61-8037-7371f729fa34      web   
996  8bc047ac-dc03-4a61-8037-7371f729fa34      web   
997  3f78ac76-6f81-43ec-85e8-f3cf74fc8fdc      web   
998  3f78ac76-6f81-43ec-85e8-f3cf74fc8fdc      web   
999  1408b7ab-6450-4337-b57d-fd8a64b7fe97      web   

                                            user_agent                 date  
0    Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...  2023-03-01T13:29:22  
1    Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...  2023-03-01T16:44:28  
2    Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...  2023-03-06T06:12:36  
3    

In [9]:
load_dotenv()
DATE_BEGIN = os.getenv('DATE_BEGIN')
DATE_END = os.getenv('DATE_END')
API_URL = os.getenv('API_URL')
print(API_URL)

None


In [8]:
response = requests.get(f'{API_URL}/registrations?begin={DATE_BEGIN}&end={DATE_END}')
data = response.json()
reg_data = pd.DataFrame(data)
reg_data.head(100)

MissingSchema: Invalid URL 'None/registrations?begin=None&end=None': No scheme supplied. Perhaps you meant https://None/registrations?begin=None&end=None?

In [ ]:
response = requests.get(f'{API_URL}/visits?begin={DATE_BEGIN}&end={DATE_END}')
data = response.json()
visits_data = pd.DataFrame(data)
visits_data.head(200)

In [ ]:
group_reg = reg_data.copy()
group_reg['datetime'] = pd.to_datetime(group_reg['datetime']).dt.date
group_reg = group_reg.groupby(['datetime', 'platform'])\
                     .size()\
                     .reset_index(name='registrations')\
                     .rename(columns={'datetime':'date_group'})
group_reg.head(100)

In [ ]:
group_visits = visits_data.copy()
group_visits['datetime'] = pd.to_datetime(group_visits['datetime']).dt.date
group_visits_filter = group_visits.sort_values('datetime').groupby('visit_id', as_index=False).last()
group_visits_filter = group_visits_filter[group_visits_filter['platform'] != 'bot'].groupby(['datetime', 'platform']) \
                                         .size() \
                                         .reset_index(name='visits') \
                                         .rename(columns={'datetime':'date_group'})
group_visits_filter.head(100)
#print(group_visits_filter['platform'].unique())

In [ ]:
conversion_data = pd.merge(
    group_visits_filter,
    group_reg,
    on=['date_group', 'platform'],
    how='left'
)
conversion_data['conversion'] = conversion_data['registrations'] / conversion_data['visits'] * 100
conversion_data['registrations'] = conversion_data['registrations'].fillna(0).astype(int)
conversion_data.to_json('./conversion.json')
print(conversion_data)

In [ ]:
response = requests.get('https://drive.usercontent.google.com/u/0/uc?id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj&export=download')
ads_csv = pd.read_csv(StringIO(response.text))
ads_csv.head(10)

In [ ]:
ads_data = ads_csv.copy()
ads_data['date'] = pd.to_datetime(ads_data['date']).dt.date

visits_reg = conversion_data.copy()
visits_reg = visits_reg.groupby('date_group', as_index=False)[['visits', 'registrations']].sum()
visits_reg.head()

visits_registrations_ads = pd.merge(
    visits_reg,
    ads_data,
    left_on='date_group',
    right_on='date',
    how='left'
)

visits_registrations_ads = visits_registrations_ads[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']]
visits_registrations_ads = visits_registrations_ads.fillna({'cost':0, 'utm_campaign':'none'})

visits_registrations_ads.to_json('./ads.json')
visits_registrations_ads.head()

In [ ]:
!mkdir charts
START_DATE = '2023-03-01'
END_DATE = '2023-04-01'

In [ ]:
vis_reg_ads_visual = visits_registrations_ads.copy()
vis_reg_ads_visual['date_group'] = vis_reg_ads_visual['date_group'].astype(str)
vis_reg_ads_visual =  vis_reg_ads_visual[vis_reg_ads_visual['date_group'].between(START_DATE, END_DATE)]

ax = vis_reg_ads_visual.plot(
    x='date_group', 
    y='visits', 
    kind='bar', 
    figsize=(max(10, len(vis_reg_ads_visual) * 0.5), 6),  # динамический размер
    color='steelblue',  # добавляем цвет
    edgecolor='navy',   # обводка
    alpha=0.8           # прозрачность для красоты
)

ax.bar_label(ax.containers[0], # подписи значений
             padding=3, 
             fontsize=10)

ax.set_title(f'Визиты с {START_DATE} по {END_DATE}', # Название графика
             fontsize=16, 
             pad=20,  # отступ заголовка
             fontweight='bold')

# названия осей
ax.set_xlabel('Дата', fontsize=12)
ax.set_ylabel('Количество визитов', fontsize=12)

# подписи осей
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', labelsize=10)

# cетка для лучшей читаемости
ax.grid(axis='y', alpha=0.3, linestyle='--')

# убрали легенду
ax.legend().remove()

# Автоматическая подгонка
#plt.tight_layout()

plt.savefig('./charts/visits.png', bbox_inches='tight') # сохранение с автоматическим размером
plt.show()

In [ ]:
df_viz = conversion_data.copy()
df_viz['date_group'] = df_viz['date_group'].astype(str)
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)]
pivot_df = df_viz.pivot_table(
          index='date_group', 
          columns='platform', 
          values='visits',
          aggfunc='sum'
      )
ax = pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(max(10, len(vis_reg_ads_visual) * 0.5), 6)
)
ax.set_title(f'Визиты с {START_DATE} по {END_DATE}', # Название графика
             fontsize=16, 
             pad=20,  # отступ заголовка
             fontweight='bold')
ax.set_xlabel('Дата', fontsize=12)
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.set_ylabel('Количество визитов', fontsize=12)
plt.savefig('./charts/visits_by_platform.png', bbox_inches='tight')
plt.show()

In [ ]:
df_viz = visits_registrations_ads.copy()
df_viz['date_group'] = df_viz['date_group'].astype(str)
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)]
ax = vis_reg_ads_visual.plot(
    x='date_group', 
    y='registrations', 
    kind='bar', 
    figsize=(max(10, len(vis_reg_ads_visual) * 0.5), 6),  # динамический размер
    color='steelblue',  # добавляем цвет
    edgecolor='navy',   # обводка
    alpha=0.8           # прозрачность для красоты
)
ax.bar_label(ax.containers[0], # подписи значений
             padding=3, 
             fontsize=10)
ax.set_title(f'Регистрации с {START_DATE} по {END_DATE}', # Название графика
             fontsize=16, 
             pad=20,  # отступ заголовка
             fontweight='bold')
# названия осей
ax.set_xlabel('Дата', fontsize=12)
ax.set_ylabel('Количество регистраций', fontsize=12)
# подписи осей
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', labelsize=10)
# 6. Сетка для лучшей читаемости
ax.grid(axis='y', alpha=0.3, linestyle='--')
# убрали легенду
ax.legend().remove()
# Автоматическая подгонка
#plt.tight_layout()
plt.savefig('./charts/registrations.png', bbox_inches='tight')
plt.show()

In [ ]:
df_viz = conversion_data.copy()
df_viz['date_group'] = df_viz['date_group'].astype(str)
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)]

pivot_df = df_viz.pivot_table(
          index='date_group', 
          columns='platform', 
          values='registrations',
          aggfunc='sum'
      )

ax = pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(max(10, len(vis_reg_ads_visual) * 0.5), 6)
)

ax.set_title(f'Регистрации с {START_DATE} по {END_DATE}', # Название графика
             fontsize=16, 
             pad=20,  # отступ заголовка
             fontweight='bold')

ax.set_xlabel('Дата', fontsize=12)
ax.tick_params(axis='x', rotation=45, labelsize=10)

ax.set_ylabel('Количество регистраций', fontsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.savefig('./charts/registrations_by_platfom.png', bbox_inches='tight')
plt.show()

In [ ]:
df_viz = conversion_data.copy()
df_viz['date_group'] = pd.to_datetime(df_viz['date_group'])  # ← убрал .astype(str)!
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)] \
                                    .groupby('date_group', as_index=False)['conversion'] \
                                    .mean()

# Теперь date_group - это datetime, а не строка
ax = df_viz.plot(x='date_group',
                 y='conversion',
                 figsize=(max(10, len(df_viz) * 0.5), 6),
                 marker='o',
                 linewidth=1,
                 markersize=6,
                 color='green',
         )

ax.set_title(f'Конверсии с {START_DATE} по {END_DATE}', 
             fontsize=16, pad=20, fontweight='bold')

for x, y in zip(df_viz['date_group'], df_viz['conversion']): # подписи знчений
    ax.annotate(f'{y:.1f}%', 
                (x, y), 
                textcoords="offset points", 
                xytext=(0,10), 
                ha='center',
                fontsize=9)

ax.set_xlabel('Дата', fontsize=12)
ax.set_ylabel('Конверсии', fontsize=12)
ax.legend().remove()

# Форматируем ось Y в проценты
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig('./charts/conversion.png', bbox_inches='tight')
plt.show()

In [ ]:
df_viz = conversion_data.copy()
df_viz['date_group'] = df_viz['date_group'].astype(str)
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)] 

fig = px.line(df_viz, 
              x='date_group', 
              y='conversion',
              color='platform',
              text=df_viz['conversion'].apply(lambda x: f'{x:.0f}'),
              markers=True,
              width=1900, 
              height=500,
        )

fig.update_layout(
    yaxis_tickformat='.2f',
    xaxis_title='Дата',
    yaxis_title='Конверсии',
    title='Конверсии по платформам'
)

fig.update_traces(
    textposition='bottom center',  # позиция: top center, bottom center, top left и т.д.
    textfont=dict(size=10) 
)

fig.write_image('./charts/conversion_by_platform.png', width=1900, height=500, scale=2)
fig.show()

In [ ]:
df_viz = visits_registrations_ads.copy()
df_viz['date_group'] = df_viz['date_group'].astype(str)
df_viz = df_viz[df_viz['date_group'].between(START_DATE, END_DATE)] 

fig = px.line(df_viz, 
              x='date_group', 
              y=['cost', 'visits'],
              markers=True,
              width=1900, 
              height=500,
        )

# Настраиваем подписи
fig.update_traces(
    name='Затраты',
    selector={'name': 'cost'},
    text=df_viz['cost'].apply(lambda x: f'{x:.0f} ₽'),
    textposition='bottom center',
    textfont=dict(size=9, color='blue'),
    mode='lines+markers+text'  # включаем отображение текста!
)

# Настраиваем подписи
fig.update_traces(
    name='Визиты',
    selector={'name': 'visits'},
    text=df_viz['visits'].apply(lambda x: f'{x:.0f}'),
    textposition='top center',
    textfont=dict(size=9, color='red'),
    mode='lines+markers+text'
)

fig.update_layout(
    yaxis_tickformat='.2f',
    xaxis_title='Дата',
    yaxis_title='',
    title='Расходы на рекламу / визиты'
)

fig.write_image('./charts/visits_and_cost.png', width=1900, height=500, scale=1)
fig.show()

In [ ]:
# Создаем подграфики
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Получаем уникальные кампании
unique_campaigns = visits_registrations_ads['utm_campaign'].unique()

# Добавляем график визитов
fig.add_trace(
    go.Scatter(x=visits_registrations_ads['date_group'], 
               y=visits_registrations_ads['visits'],
               mode='lines+markers',
               name='Визиты',
               line=dict(color='black', width=2),
               marker=dict(size=6)),
    secondary_y=False
)

# Добавляем столбцы стоимости рекламы
fig.add_trace(
    go.Bar(x=visits_registrations_ads['date_group'], 
           y=visits_registrations_ads['cost'],
           name='Стоимость рекламы',
           marker_color='gray',
           opacity=0.3),
    secondary_y=True
)

# Добавляем цветовые области для кампаний
for campaign in unique_campaigns:
    if campaign != 'none':
        campaign_data = visits_registrations_ads[visits_registrations_ads['utm_campaign'] == campaign]
        for _, row in campaign_data.iterrows():
            fig.add_vrect(
                x0=row['date_group'] - pd.Timedelta(days=0.5),
                x1=row['date_group'] + pd.Timedelta(days=0.5),
                fillcolor= f'rgba({hash(campaign) % 255}, {(hash(campaign) * 2) % 255}, {(hash(campaign) * 3) % 255}, 0.2)',
                opacity=0.3,
                line_width=0,
                name=f'Кампания: {campaign}' if _ == campaign_data.index[0] else None
            )

# Настройка layout
fig.update_layout(
    title='Визиты с выделением рекламных кампаний',
    xaxis_title='Дата',
    hovermode='x unified',
    showlegend=True,
    height=500
)

# Настройка осей
fig.update_yaxes(title_text="Количество визитов", secondary_y=False)
fig.update_yaxes(title_text="Стоимость рекламы", secondary_y=True)

# Показываем график
fig.show()